In [ ]:
from herbie import Herbie, FastHerbie
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from itertools import chain

In [ ]:
h = Herbie(
  '2021-01-01 12:00',
  model='hrrr',
  product='prs',
  fxx=3
)

In [ ]:
h.inventory()

In [ ]:
tmp = h.xarray("TMP:2 m")

In [ ]:
plt.imshow(tmp.t2m)

In [ ]:
hgt = h.xarray("HGT:surface")

In [ ]:
type(hgt)

In [ ]:
hgt

In [ ]:
plt.imshow(hgt.orog, origin='lower')

In [ ]:
hgt.orog.max()

In [ ]:
terr = h.terrain()

In [ ]:
plt.imshow(terr.orog, origin='lower')

In [ ]:
H2 = Herbie(
  '2024-11-05 08:00',
  model='hrrr',
  product='prs',
  fxx=3
)

In [ ]:
hgt2 = H2.xarray("HGT:surface")
terr2 = H2.terrain()

In [ ]:
np.mean(np.abs(hgt2.orog - hgt.orog))

In [ ]:
dates = pd.date_range("2024-01-01", periods=12, freq="1H")

In [ ]:
dates

In [ ]:
Hdates = FastHerbie(dates, model="hrrr", product = "sfc", fxx=[3])

In [ ]:
Hdates.file_exists

In [ ]:
H = Hdates.file_exists[0]

In [ ]:
inv = H.inventory()

In [ ]:
type(inv)

In [ ]:
inv.variable.str.contains("APCP").any()

In [ ]:
inv[inv.variable.str.contains("APCP")]

In [ ]:
# ds = [
#     H.xarray("(?:TMP:2 m|RH:2 m|APCP: surface)")
#     for H in Hdates.file_exists
# ]
ds = [
    H.xarray("(?:TMP:2 m|RH:2 m|:APCP:.*:(?:0-1|[1-9]\d*-\d+) hour)")
    for H in Hdates.file_exists
]

In [ ]:
ds[0]

In [ ]:
ds = list(chain(*ds))

In [ ]:
len(ds)

In [ ]:
def merge_datasets(ds_list):
    """Merge list of Datasets together.

    Since cfgrib doesn't merge data in different "hypercubes", we will
    do the merge ourselves.

    Parameters
    ----------
    ds_list : list
        A list of xarray.Datasets, usually from the list of datasets
        returned by cfgrib when data is on multiple levels.
    """
    these = []
    for ds in ds_list:
        ds = ds.drop_vars("gribfile_projection")
        expand_dims = []
        for i in [
            "heightAboveGround",
            "time",
            "step",
            "isobaricInhPa",
            "depthBelowLandLayer",
        ]:
            if i in ds and i not in ds.dims:
                expand_dims.append(i)
        these.append(ds.expand_dims(expand_dims))
    return xr.merge(these, compat="override")

In [ ]:
ds = merge_datasets(ds)
ds

In [ ]:
ds.t2m.dims

In [ ]:
ds.t2m.shape

In [ ]:
ds.t2m[0,:,0,0,0]

In [ ]:
ds.tp.shape

In [ ]:
ds.dims

In [ ]:
ds.tp[:,0,0,0]